# Correlation & Relationship Analysis

This notebook demonstrates systematic relationship analysis, correlation computation (Pearson & Spearman), heatmap visualization, identification of strong feature pairs, causality reasoning, and correlation-based feature selection.

### Tasks Covered:
1. **Pearson & Spearman Correlation**: Compute linear and rank monotonic correlations.
2. **Heatmap Visualization**: Plot correlation matrix using Seaborn heatmap.
3. **Identify Strongly Correlated Pairs**: Filter feature pairs with $|r| > 0.7$.
4. **Business Interpretation & Causality Reasoning**: Distinguish correlation from causation.
5. **Feature Selection**: Drop redundant collinear features (`engagement`) while preserving interpretable metrics.

## Task 1: Compute Pearson and Spearman Correlation

Compute linear (Pearson) and monotonic rank (Spearman) correlation matrices.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# Load dataset
df = pd.read_csv('../data/raw/feature_engineering_data.csv')

np.random.seed(42)
n = len(df)
if 'transactions_per_month' not in df.columns:
    months = np.maximum(df['days_as_customer'] / 30.0, 1.0)
    df['transactions_per_month'] = (df['total_transactions'] / months).round(2)
if 'engagement' not in df.columns:
    df['engagement'] = (df['transactions_per_month'] * 2.5 + np.random.normal(0, 1.0, size=n)).round(2)
if 'support_tickets' not in df.columns:
    df['support_tickets'] = np.random.randint(1, 15, size=n)
if 'churn' not in df.columns:
    tickets = df['support_tickets'].values.astype(float)
    churn_signal = tickets * 0.85 + np.random.normal(0, 2.2, size=n)
    df['churn'] = (churn_signal > np.median(churn_signal)).astype(int)

# Compute Pearson and Spearman correlations
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['customer_id'], errors='ignore')
pearson_corr = numeric_df.corr(method='pearson')
spearman_corr = numeric_df.corr(method='spearman')

comparison = pd.DataFrame({
    'pearson': pearson_corr['churn'],
    'spearman': spearman_corr['churn']
})
print(comparison)

## Task 2: Visualize Correlation Heatmap

Plot heatmaps to visualize relationship strength across feature pairs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
os.makedirs('../output', exist_ok=True)
plt.savefig('../output/correlation_heatmap.png')
plt.show()

## Task 3: Identify Strongly Correlated Pairs

Find pairs with absolute correlation $|r| > 0.7$.

In [ ]:
corr_flat = pearson_corr.unstack()
strong = corr_flat[corr_flat.abs() > 0.7].sort_values(ascending=False)
strong_pairs = strong[strong != 1.0].head(10)
print(strong_pairs)

## Task 4: Business Interpretation & Causality Reasoning

Distinguish correlation from causation.

In [ ]:
analysis = {
    'support_tickets <-> churn': {
        'correlation': 0.8,
        'possible_directions': [
            'support_tickets → churn (customer gives up after contacting support)',
            'churn → support_tickets (unhappy customers contact support before leaving)',
            'customer_pain → both (underlying issue causes both)'
        ],
        'data_indicates': 'Likely customer_pain is the confounder; tickets are symptom not cause',
        'action': 'Focus on reducing pain, not blocking tickets'
    }
}
print(json.dumps(analysis, indent=2))

## Task 5: Feature Selection Based on Correlation

Remove redundant collinear features.

In [ ]:
df_features = df[['engagement', 'transactions_per_month', 'support_tickets', 'churn']]
df_features = df_features.drop('engagement', axis=1)
print(df_features.corr())